# FujiCV — CIFAR-10 Training on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dsabarinathan/fujicv/blob/main/examples/colab_cifar10.ipynb)

Trains a **ResNet-18** on **CIFAR-10** (10 classes) using the `fujicv` package.  
Results — plots, classification report, confusion matrix — are saved to `/content/results/`.

**Runtime:** GPU (T4) recommended → Runtime → Change runtime type → T4 GPU

## 1. Install dependencies

In [ ]:
!pip install fujicv timm albumentations iterative-stratification -q
print('Installation complete.')

## 2. Imports & config

In [ ]:
import logging
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

import fujicv
from fujicv.data.datasets import get_default_dataset
from fujicv.data.transforms import get_train_transforms, get_val_transforms
from fujicv.engine.trainer import Trainer
from fujicv.eval.curves import plot_pr_curve, plot_roc_curve
from fujicv.eval.plots import plot_loss_curves, plot_metric_curves
from fujicv.eval.report import classification_report
from fujicv.losses.classification import CrossEntropyLoss
from fujicv.metrics.classification import Accuracy, F1
from fujicv.models.builder import ModelBuilder
from fujicv.utils.seed import set_seed

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

# ── Config ────────────────────────────────────────────────────────────────────
SEED       = 42
BACKBONE   = 'resnet18'
IMAGE_SIZE = 32
BATCH_SIZE = 256
EPOCHS     = 10
LR         = 1e-3
OUTPUT_DIR = '/content/results'
DATA_DIR   = '/content/data'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

set_seed(SEED)
device = fujicv.get_device()
print(f'Device: {device}')

## 3. Load CIFAR-10 dataset

In [ ]:
train_ds, val_ds, class_to_idx = get_default_dataset(
    name='cifar10',
    root=DATA_DIR,
    train_transform=get_train_transforms(IMAGE_SIZE, level='medium'),
    val_transform=get_val_transforms(IMAGE_SIZE),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

idx_to_class = {v: k for k, v in class_to_idx.items()}
class_names  = [idx_to_class[i] for i in range(len(class_to_idx))]

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,}')
print(f'Classes: {class_names}')

## 4. Preview a batch

In [ ]:
imgs, labels = next(iter(val_loader))

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    img = imgs[i].permute(1, 2, 0).numpy()
    img = np.clip(img * std + mean, 0, 1)
    ax.imshow(img)
    ax.set_title(class_names[labels[i].item()], fontsize=8)
    ax.axis('off')
plt.suptitle('CIFAR-10 Sample Batch', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sample_batch.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → /content/results/sample_batch.png')

## 5. Build model

In [ ]:
model = ModelBuilder(
    backbone_name=BACKBONE,
    backbone_source='timm',
    pretrained=True,
    task='classification',
    num_outputs=10,
    image_size=IMAGE_SIZE,
).build()

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model: {BACKBONE} | Parameters: {total_params:.1f}M')

## 6. Train

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=CrossEntropyLoss(),
    metrics={'accuracy': Accuracy(), 'f1': F1()},
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=EPOCHS,
    task='classification',
    output_dir=OUTPUT_DIR,
    class_to_idx=class_to_idx,
    monitor_metric='val_accuracy',
    mixed_precision=True,
    early_stopping_patience=5,
)

history = trainer.train()

## 7. Training curves

In [ ]:
fig = plot_loss_curves(history)
fig.savefig(f'{OUTPUT_DIR}/loss_curves.png', dpi=120, bbox_inches='tight')
plt.show()

fig = plot_metric_curves(history, 'accuracy')
fig.savefig(f'{OUTPUT_DIR}/accuracy_curves.png', dpi=120, bbox_inches='tight')
plt.show()

fig = plot_metric_curves(history, 'f1')
fig.savefig(f'{OUTPUT_DIR}/f1_curves.png', dpi=120, bbox_inches='tight')
plt.show()

print('Saved → loss_curves.png, accuracy_curves.png, f1_curves.png')

## 8. Evaluate on validation set

In [ ]:
model.eval()
all_preds, all_probs, all_targets = [], [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(preds)
        all_targets.append(labels.numpy())

all_probs   = np.concatenate(all_probs)
all_preds   = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

acc = (all_preds == all_targets).mean()
print(f'Val Accuracy: {acc:.4f} ({acc*100:.2f}%)')

## 9. Classification report & confusion matrix

In [ ]:
report_text, fig_cm = classification_report(all_targets, all_preds, class_names)
print(report_text)

with open(f'{OUTPUT_DIR}/classification_report.txt', 'w') as f:
    f.write(report_text)

fig_cm.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → classification_report.txt, confusion_matrix.png')

## 10. ROC & PR curves

In [ ]:
fig_roc = plot_roc_curve(all_targets, all_probs, class_names, multi_class='ovr')
fig_roc.savefig(f'{OUTPUT_DIR}/roc_curve.png', dpi=120, bbox_inches='tight')
plt.show()

fig_pr = plot_pr_curve(all_targets, all_probs, class_names)
fig_pr.savefig(f'{OUTPUT_DIR}/pr_curve.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → roc_curve.png, pr_curve.png')

## 11. t-SNE embedding visualization

In [ ]:
from fujicv.eval.tsne import extract_embeddings, plot_tsne

# Use a subset for speed
subset = torch.utils.data.Subset(val_ds, range(2000))
subset_loader = DataLoader(subset, batch_size=256, shuffle=False, num_workers=2)

embeddings, embed_labels = extract_embeddings(model, subset_loader, device)
fig_tsne = plot_tsne(embeddings, embed_labels, class_names)
fig_tsne.savefig(f'{OUTPUT_DIR}/tsne.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → tsne.png')

## 12. Summary

In [ ]:
best_acc = max(history.metrics.get('val_accuracy', [0]))
best_f1  = max(history.metrics.get('val_f1', [0]))

summary = f"""
========================================
 FujiCV CIFAR-10 Training Summary
========================================
 Model      : {BACKBONE} (pretrained)
 Dataset    : CIFAR-10 (10 classes)
 Epochs     : {EPOCHS}
 Device     : {device}
 Batch size : {BATCH_SIZE}
----------------------------------------
 Best Val Accuracy : {best_acc:.4f} ({best_acc*100:.2f}%)
 Best Val F1       : {best_f1:.4f}
----------------------------------------
 Results saved to  : {OUTPUT_DIR}/
   sample_batch.png
   loss_curves.png
   accuracy_curves.png
   f1_curves.png
   confusion_matrix.png
   classification_report.txt
   roc_curve.png
   pr_curve.png
   tsne.png
   best.pt  (best checkpoint)
   history.csv
========================================
"""
print(summary)

with open(f'{OUTPUT_DIR}/summary.txt', 'w') as f:
    f.write(summary)

# List all saved files
print('Files in /content/results:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1024
    print(f'  {f:<35} {size:6.1f} KB')

## 13. Download results (optional)

Run the cell below to zip and download all results to your local machine.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/fujicv_cifar10_results', 'zip', OUTPUT_DIR)
files.download('/content/fujicv_cifar10_results.zip')
print('Downloaded fujicv_cifar10_results.zip')